# 1. Imports

In [168]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import OneHotEncoder, TargetEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import RandomizedSearchCV
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, root_mean_squared_error, r2_score

# 2. Preprocessed Data Import

In [130]:
df_preprocessed = pd.read_csv(filepath_or_buffer='../data/preprocessed/dataset_preprocesado.csv')
df_preprocessed.head()

,tipo_de_entrada,tema_solicitud,sexo,alcaldia_solicitud,dia_solicitud,mes_solicitud,hora_sin,hora_cos,tiempo_resolución,tiempo_resolución_con_hora
0,Portal Ciudadano SUAC,FALTA DE AGUA,No especificado,Gustavo A. Madero,Wednesday,January,1.224647e-16,-1.000000,7.0,6.468537
1,Portal Ciudadano SUAC,FALTA DE AGUA,No especificado,Gustavo A. Madero,Wednesday,January,-5.000000e-01,-0.866025,7.0,6.375353
2,Portal Ciudadano SUAC,FALTA DE AGUA,No especificado,Gustavo A. Madero,Wednesday,January,-8.660254e-01,-0.500000,7.0,6.305584
3,Portal Ciudadano SUAC,FALTA DE AGUA,No especificado,Gustavo A. Madero,Wednesday,January,-9.659258e-01,-0.258819,7.0,6.265206
4,Portal Ciudadano SUAC,SOLICITUD DE VIGILANCIA POLICIACA,No especificado,Gustavo A. Madero,Wednesday,January,-9.659258e-01,0.258819,7.0,6.189034


# 3. Data Splitting

In [131]:
X = df_preprocessed.drop(columns=['tiempo_resolución_con_hora','tiempo_resolución'])
y = df_preprocessed['tiempo_resolución_con_hora']

In [132]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 4. Encoding

## 4.1 Encoding "Tema"

### 4.1.1 Obtain 80% of topics

In [133]:
X_train.info()

<class 'pandas.DataFrame'>
Index: 1437825 entries, 102925 to 121958
Data columns (total 8 columns):
 #   Column              Non-Null Count    Dtype  
---  ------              --------------    -----  
 0   tipo_de_entrada     1437825 non-null  str    
 1   tema_solicitud      1437825 non-null  str    
 2   sexo                1437825 non-null  str    
 3   alcaldia_solicitud  1437825 non-null  str    
 4   dia_solicitud       1437825 non-null  str    
 5   mes_solicitud       1437825 non-null  str    
 6   hora_sin            1437825 non-null  float64
 7   hora_cos            1437825 non-null  float64
dtypes: float64(2), str(6)
memory usage: 207.2 MB


In [134]:
conteo_tema = X_train['tema_solicitud'].value_counts(normalize=True)
conteo_tema

tema_solicitud
OTRO                                               0.163548
FALTA DE AGUA                                      0.114986
REPARACIÓN Y MANTENIMIENTO DE ALUMBRADO PÚBLICO    0.109045
PODA                                               0.087147
FUGA DE AGUA                                       0.071577
                                                     ...   
RETIRO ÁRBOL                                       0.000015
COMEDORES COMUNITARIOS / PÚBLICOS                  0.000014
INFORMACIÓN, PROGRAMAS Y EVENTOS PARA JÓVENES      0.000013
RECICLAJE                                          0.000013
JORNADAS NOTARIALES                                0.000011
Name: proportion, Length: 72, dtype: float64

In [135]:
frecuencia_acumulada = conteo_tema.drop(labels='OTRO').cumsum()
frecuencia_acumulada

tema_solicitud
FALTA DE AGUA                                      0.114986
REPARACIÓN Y MANTENIMIENTO DE ALUMBRADO PÚBLICO    0.224031
PODA                                               0.311178
FUGA DE AGUA                                       0.382755
BACHEO                                             0.444010
                                                     ...   
RETIRO ÁRBOL                                       0.836401
COMEDORES COMUNITARIOS / PÚBLICOS                  0.836415
INFORMACIÓN, PROGRAMAS Y EVENTOS PARA JÓVENES      0.836428
RECICLAJE                                          0.836440
JORNADAS NOTARIALES                                0.836452
Name: proportion, Length: 71, dtype: float64

In [136]:
top_categorias = frecuencia_acumulada.where(frecuencia_acumulada <= 0.8).dropna().index.to_list()
top_categorias

['FALTA DE AGUA',
 'REPARACIÓN Y MANTENIMIENTO DE ALUMBRADO PÚBLICO',
 'PODA',
 'FUGA DE AGUA',
 'BACHEO',
 'DESAZOLVE',
 'MANTENIMIENTO VÍA PÚBLICA',
 'SOLICITUD DE VIGILANCIA POLICIACA',
 'VERIFICACIÓN ADMINISTRATIVA',
 'RETIRO CASCAJO, ESCOMBRO, AZOLVE, RAMAS',
 'VEHÍCULO ABANDONADO / CHATARRIZACIÓN',
 'MANTENIMIENTO DE COLADERA / ALCANTARILLA',
 'QUEJA DE TRANSPORTE PÚBLICO',
 'BARBECHO / CHAPONEO',
 'MANTENIMIENTO DRENAJE',
 'RECOLECCIÓN BASURA',
 'LIMPIEZA VÍA PÚBLICA',
 'MANTENIMIENTO PARQUE / ÁREA VERDE',
 'PAVIMENTACIÓN / REENCARPETADO',
 'QUEJA SERVIDOR PÚBLICO',
 'MANTENIMIENTO DE BANQUETAS',
 'RETIRO AMBULANTE',
 'MANTENIMIENTO SEMÁFOROS',
 'PROTECCIÓN CIVIL',
 'RECUPERACIÓN DE VIALIDADES',
 'USO DE SUELO',
 'TRÁMITES / INFO MOVILIDAD']

### 4.1.2 Create Compact Topic Column

In [137]:
X_train['tema_solicitud_compacto'] = X_train['tema_solicitud'].where(X_train['tema_solicitud'].isin(top_categorias),other='OTRO')
print(X_train['tema_solicitud_compacto'].value_counts(normalize=True).cumsum())

tema_solicitud_compacto
OTRO                                               0.200166
FALTA DE AGUA                                      0.315152
REPARACIÓN Y MANTENIMIENTO DE ALUMBRADO PÚBLICO    0.424198
PODA                                               0.511345
FUGA DE AGUA                                       0.582921
BACHEO                                             0.644176
DESAZOLVE                                          0.701551
MANTENIMIENTO VÍA PÚBLICA                          0.739155
SOLICITUD DE VIGILANCIA POLICIACA                  0.765976
VERIFICACIÓN ADMINISTRATIVA                        0.792093
RETIRO CASCAJO, ESCOMBRO, AZOLVE, RAMAS            0.817161
VEHÍCULO ABANDONADO / CHATARRIZACIÓN               0.841515
MANTENIMIENTO DE COLADERA / ALCANTARILLA           0.865845
QUEJA DE TRANSPORTE PÚBLICO                        0.882945
BARBECHO / CHAPONEO                                0.898715
MANTENIMIENTO DRENAJE                              0.912980
RECOLECCIÓN BASU

In [138]:
X_test['tema_solicitud_compacto'] = X_test['tema_solicitud'].where(X_test['tema_solicitud'].isin(top_categorias),other='OTRO')

### 4.1.3 Drop Column

In [139]:
X_train.drop(columns=['tema_solicitud'],inplace=True)
X_train

,tipo_de_entrada,sexo,alcaldia_solicitud,dia_solicitud,mes_solicitud,hora_sin,hora_cos,tema_solicitud_compacto
102925,Portal Ciudadano SUAC,Masculino,Iztacalco,Friday,February,7.071068e-01,-7.071068e-01,FALTA DE AGUA
148356,Portal Ciudadano SUAC,No especificado,Azcapotzalco,Friday,March,-7.071068e-01,-7.071068e-01,FALTA DE AGUA
1608369,App CDMX,No especificado,Cuauhtémoc,Wednesday,December,-1.000000e+00,-1.836970e-16,PODA
370152,Portal Ciudadano SUAC,Masculino,Milpa Alta,Wednesday,August,1.224647e-16,-1.000000e+00,FALTA DE AGUA
22356,Interoperabilidad Iztapalapa,Femenino,Iztapalapa,Monday,October,-9.659258e-01,2.588190e-01,OTRO
...,...,...,...,...,...,...,...,...
259178,Portal Ciudadano SUAC,Masculino,Azcapotzalco,Monday,June,1.224647e-16,-1.000000e+00,OTRO
1414414,Operador Telefónico - Ventanilla,No especificado,Gustavo A. Madero,Thursday,September,-8.660254e-01,-5.000000e-01,VERIFICACIÓN ADMINISTRATIVA
131932,Usuarios SUAC Dependencia,No especificado,Iztacalco,Wednesday,March,5.000000e-01,-8.660254e-01,MANTENIMIENTO DE COLADERA / ALCANTARILLA
671155,App CDMX,No especificado,Benito Juárez,Monday,September,-5.000000e-01,8.660254e-01,BACHEO


In [140]:
X_test.drop(columns=['tema_solicitud'],inplace=True)
X_test

,tipo_de_entrada,sexo,alcaldia_solicitud,dia_solicitud,mes_solicitud,hora_sin,hora_cos,tema_solicitud_compacto
97447,Portal Ciudadano SUAC,No especificado,Álvaro Obregón,Monday,February,-8.660254e-01,-0.500000,BACHEO
1199308,App CDMX,No especificado,Coyoacán,Monday,June,5.000000e-01,-0.866025,MANTENIMIENTO DRENAJE
779928,Operador Telefónico LOCATEL,Masculino,Cuauhtémoc,Friday,December,9.659258e-01,-0.258819,QUEJA DE TRANSPORTE PÚBLICO
838957,Interoperabilidad Iztapalapa,Femenino,Iztapalapa,Wednesday,January,1.224647e-16,-1.000000,OTRO
646220,Portal Ciudadano SUAC,Masculino,Iztacalco,Sunday,September,-8.660254e-01,0.500000,DESAZOLVE
...,...,...,...,...,...,...,...,...
1300968,App CDMX,No especificado,Coyoacán,Monday,July,2.588190e-01,-0.965926,FALTA DE AGUA
1143907,Operador Telefónico LOCATEL,Femenino,Gustavo A. Madero,Friday,May,-7.071068e-01,-0.707107,OTRO
1068157,Usuarios SUAC Dependencia,Femenino,La Magdalena Contreras,Friday,March,5.000000e-01,-0.866025,DESAZOLVE
1311390,Interoperabilidad SACMEX,No especificado,Cuauhtémoc,Thursday,July,-8.660254e-01,0.500000,MANTENIMIENTO DE COLADERA / ALCANTARILLA


## 4.2 Encode Categorical Columns

In [141]:
columnas_ohe = ['tipo_de_entrada','sexo','alcaldia_solicitud','dia_solicitud','mes_solicitud','tema_solicitud_compacto']
ohe = OneHotEncoder(sparse_output=False,handle_unknown='ignore')
for param in columnas_ohe:
    param_encoded = ohe.fit_transform(X_train[[param]])
    param_columns = ohe.get_feature_names_out([param])
    X_train[param_columns] = param_encoded
    X_train.drop(columns=[param],inplace=True)
    param_encoded_test = ohe.transform(X_test[[param]])
    X_test[param_columns] = param_encoded_test
    X_test.drop(columns=[param],inplace=True)
X_train

,hora_sin,hora_cos,tipo_de_entrada_App CDMX,tipo_de_entrada_Interoperabilidad Iztapalapa,tipo_de_entrada_Interoperabilidad Miguel Hidalgo,tipo_de_entrada_Interoperabilidad SACMEX,tipo_de_entrada_Interoperabilidad Xochimilco,tipo_de_entrada_Operador Telefónico - Ventanilla,tipo_de_entrada_Operador Telefónico LOCATEL,tipo_de_entrada_Portal Ciudadano SUAC,...,tema_solicitud_compacto_RECOLECCIÓN BASURA,tema_solicitud_compacto_RECUPERACIÓN DE VIALIDADES,tema_solicitud_compacto_REPARACIÓN Y MANTENIMIENTO DE ALUMBRADO PÚBLICO,tema_solicitud_compacto_RETIRO AMBULANTE,"tema_solicitud_compacto_RETIRO CASCAJO, ESCOMBRO, AZOLVE, RAMAS",tema_solicitud_compacto_SOLICITUD DE VIGILANCIA POLICIACA,tema_solicitud_compacto_TRÁMITES / INFO MOVILIDAD,tema_solicitud_compacto_USO DE SUELO,tema_solicitud_compacto_VEHÍCULO ABANDONADO / CHATARRIZACIÓN,tema_solicitud_compacto_VERIFICACIÓN ADMINISTRATIVA
102925,7.071068e-01,-7.071068e-01,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
148356,-7.071068e-01,-7.071068e-01,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1608369,-1.000000e+00,-1.836970e-16,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
370152,1.224647e-16,-1.000000e+00,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
22356,-9.659258e-01,2.588190e-01,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
259178,1.224647e-16,-1.000000e+00,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1414414,-8.660254e-01,-5.000000e-01,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
131932,5.000000e-01,-8.660254e-01,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
671155,-5.000000e-01,8.660254e-01,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [142]:
X_train.info()

<class 'pandas.DataFrame'>
Index: 1437825 entries, 102925 to 121958
Data columns (total 78 columns):
 #   Column                                                                   Non-Null Count    Dtype  
---  ------                                                                   --------------    -----  
 0   hora_sin                                                                 1437825 non-null  float64
 1   hora_cos                                                                 1437825 non-null  float64
 2   tipo_de_entrada_App CDMX                                                 1437825 non-null  float64
 3   tipo_de_entrada_Interoperabilidad Iztapalapa                             1437825 non-null  float64
 4   tipo_de_entrada_Interoperabilidad Miguel Hidalgo                         1437825 non-null  float64
 5   tipo_de_entrada_Interoperabilidad SACMEX                                 1437825 non-null  float64
 6   tipo_de_entrada_Interoperabilidad Xochimilco                  

## 4.3 Encode Zip Code (Pending)

In [143]:
#target_encode = TargetEncoder()
#zc_encoded = target_encode.fit_transform(X_train[['codigo_postal_solicitud']],y_train)
#X_train['zc_encoded'] = zc_encoded
#X_train.drop(columns=['codigo_postal_solicitud'],inplace=True)
#X_train

In [144]:
#X_train.info()

In [145]:
#zc_encoded_test = target_encode.transform(X_test[['codigo_postal_solicitud']])
#X_test['zc_encoded'] = zc_encoded_test
#X_test.drop(columns=['codigo_postal_solicitud'],inplace=True)

In [146]:
#X_test.info()

# 5. Transformations

## 5.1 X Scaling

In [147]:
sc = StandardScaler()
train_hora_sc = sc.fit_transform(X_train[['hora_sin','hora_cos']])
X_train[['hora_sin','hora_cos']] = train_hora_sc
test_hora_sc = sc.transform(X_test[['hora_sin','hora_cos']])
X_test[['hora_sin','hora_cos']] = test_hora_sc

## 5.2 Y Transform

In [148]:
y_train_log = np.log1p(y_train)
y_test_log = np.log1p(y_test)

# 6. Feature Importance Validation (Random Forest)

In [149]:
X_train.info()

<class 'pandas.DataFrame'>
Index: 1437825 entries, 102925 to 121958
Data columns (total 78 columns):
 #   Column                                                                   Non-Null Count    Dtype  
---  ------                                                                   --------------    -----  
 0   hora_sin                                                                 1437825 non-null  float64
 1   hora_cos                                                                 1437825 non-null  float64
 2   tipo_de_entrada_App CDMX                                                 1437825 non-null  float64
 3   tipo_de_entrada_Interoperabilidad Iztapalapa                             1437825 non-null  float64
 4   tipo_de_entrada_Interoperabilidad Miguel Hidalgo                         1437825 non-null  float64
 5   tipo_de_entrada_Interoperabilidad SACMEX                                 1437825 non-null  float64
 6   tipo_de_entrada_Interoperabilidad Xochimilco                  

In [150]:
rf_sin_log = RandomForestRegressor(n_estimators=100, max_depth=15, max_features='sqrt', n_jobs=-1, random_state=0)
rf_sin_log.fit(X_train, y_train)
importancias_sin_log = pd.Series(rf_sin_log.feature_importances_, index=X_train.columns).sort_values(ascending=False)
importancias_sin_log.head(20)

tema_solicitud_compacto_BACHEO                               0.095136
tema_solicitud_compacto_SOLICITUD DE VIGILANCIA POLICIACA    0.046633
tema_solicitud_compacto_MANTENIMIENTO VÍA PÚBLICA            0.040127
alcaldia_solicitud_Coyoacán                                  0.039096
tema_solicitud_compacto_PODA                                 0.038184
tema_solicitud_compacto_FALTA DE AGUA                        0.033788
hora_sin                                                     0.033649
hora_cos                                                     0.031740
alcaldia_solicitud_Miguel Hidalgo                            0.029889
tipo_de_entrada_Usuarios SUAC Dependencia                    0.028835
tema_solicitud_compacto_QUEJA DE TRANSPORTE PÚBLICO          0.025292
tipo_de_entrada_Interoperabilidad SACMEX                     0.023523
tipo_de_entrada_Portal Ciudadano SUAC                        0.022977
alcaldia_solicitud_Benito Juárez                             0.021850
alcaldia_solicitud_T

In [151]:
rf_con_log = RandomForestRegressor(n_estimators=100, max_depth=15, max_features='sqrt', n_jobs=-1, random_state=0)
rf_con_log.fit(X_train, y_train_log)
importancias_con_log = pd.Series(rf_con_log.feature_importances_, index=X_train.columns).sort_values(ascending=False)
importancias_con_log.head(20)

tema_solicitud_compacto_OTRO                                               0.071585
tema_solicitud_compacto_BACHEO                                             0.066821
tipo_de_entrada_Interoperabilidad SACMEX                                   0.054186
tema_solicitud_compacto_SOLICITUD DE VIGILANCIA POLICIACA                  0.052430
alcaldia_solicitud_Coyoacán                                                0.046315
tipo_de_entrada_Interoperabilidad Iztapalapa                               0.045692
alcaldia_solicitud_Tláhuac                                                 0.037312
tema_solicitud_compacto_FALTA DE AGUA                                      0.036158
tipo_de_entrada_Usuarios SUAC Dependencia                                  0.033743
hora_cos                                                                   0.024771
tema_solicitud_compacto_QUEJA DE TRANSPORTE PÚBLICO                        0.024647
alcaldia_solicitud_Gustavo A. Madero                                       0

In [152]:
def variables_originales(nombre_columna):
    if nombre_columna in ('hora_sin', 'hora_cos'):
         return 'hora_solicitud'
    else:
        for variable in columnas_ohe:
            if nombre_columna.startswith(variable):
                return variable
        else:
            return nombre_columna

In [153]:
importancias_con_log.index = importancias_con_log.index.map(variables_originales)

In [154]:
importancias_con_log.groupby(importancias_con_log.index).sum().sort_values(ascending=False)

tema_solicitud_compacto    0.405489
alcaldia_solicitud         0.233834
tipo_de_entrada            0.200811
mes_solicitud              0.061854
hora_solicitud             0.048198
dia_solicitud              0.026769
sexo                       0.023045
dtype: float64

# 7. Linear Regression

## 7.1 Con todas las variables

In [155]:
X_train_1 = X_train.copy()
X_test_1 = X_test.copy()

In [156]:
lr_1 = LinearRegression(n_jobs=-1)
model_1 = lr_1.fit(X=X_train_1, y=y_train_log)
y_pred_1 = np.expm1(model_1.predict(X_test_1))
rmse_1 = root_mean_squared_error(y_true=y_test, y_pred=y_pred_1)
mae_1 = mean_absolute_error(y_true=y_test, y_pred=y_pred_1)
r2_1 = r2_score(y_true=y_test, y_pred=y_pred_1)
print(f'RMSE 1: {rmse_1}, MAE 1: {mae_1},  R2 1: {r2_1}')

RMSE 1: 72.74015164589018, MAE 1: 38.27922219949063,  R2 1: -0.07570299822712978


## 7.2 Sin 'sexo' ni 'dia'

In [157]:
columnas_eliminar = []
for columna in X_train.columns:
    if columna.startswith('sexo'):
        columnas_eliminar.append(columna)
    elif columna.startswith('dia'):
        columnas_eliminar.append(columna)
print(columnas_eliminar)

['sexo_Femenino', 'sexo_Masculino', 'sexo_No especificado', 'dia_solicitud_Friday', 'dia_solicitud_Monday', 'dia_solicitud_Saturday', 'dia_solicitud_Sunday', 'dia_solicitud_Thursday', 'dia_solicitud_Tuesday', 'dia_solicitud_Wednesday']


In [158]:
X_train_2 = X_train.drop(columns=columnas_eliminar)
X_test_2 = X_test.drop(columns=columnas_eliminar)

In [159]:
lr_2 = LinearRegression(n_jobs=-1)
model_2 = lr_2.fit(X=X_train_2, y=y_train_log)
y_pred_2 = np.expm1(model_2.predict(X_test_2))
rmse_2 = root_mean_squared_error(y_true=y_test, y_pred=y_pred_2)
mae_2 = mean_absolute_error(y_true=y_test, y_pred=y_pred_2)
r2_2 = r2_score(y_true=y_test, y_pred=y_pred_2)
print(f'RMSE 2: {rmse_2}, MAE 2: {mae_2}, R2 2: {r2_2}')

RMSE 2: 72.7499802656004, MAE 2: 38.30448728897669, R2 2: -0.07599371496992724


## 7.3 Sin Sin 'sexo', 'dia', 'mes' ni 'hora'

In [160]:
columnas_eliminar = []
for columna in X_train.columns:
    if columna.startswith('sexo'):
        columnas_eliminar.append(columna)
    elif columna.startswith('dia'):
        columnas_eliminar.append(columna)
    elif columna.startswith('mes'):
        columnas_eliminar.append(columna)
    elif columna.startswith('hora'):
        columnas_eliminar.append(columna)
print(columnas_eliminar)

['hora_sin', 'hora_cos', 'sexo_Femenino', 'sexo_Masculino', 'sexo_No especificado', 'dia_solicitud_Friday', 'dia_solicitud_Monday', 'dia_solicitud_Saturday', 'dia_solicitud_Sunday', 'dia_solicitud_Thursday', 'dia_solicitud_Tuesday', 'dia_solicitud_Wednesday', 'mes_solicitud_April', 'mes_solicitud_August', 'mes_solicitud_December', 'mes_solicitud_February', 'mes_solicitud_January', 'mes_solicitud_July', 'mes_solicitud_June', 'mes_solicitud_March', 'mes_solicitud_May', 'mes_solicitud_November', 'mes_solicitud_October', 'mes_solicitud_September']


In [161]:
X_train_3 = X_train.drop(columns=columnas_eliminar)
X_test_3 = X_test.drop(columns=columnas_eliminar)

In [162]:
lr_3 = LinearRegression(n_jobs=-1)
model_3 = lr_3.fit(X=X_train_3, y=y_train_log)
y_pred_3 = np.expm1(model_3.predict(X_test_3))
rmse_3 = root_mean_squared_error(y_true=y_test, y_pred=y_pred_3)
mae_3 = mean_absolute_error(y_true=y_test, y_pred=y_pred_3)
r2_3 = r2_score(y_true=y_test, y_pred=y_pred_3)
print(f'RMSE 3: {rmse_3}, MAE 3: {mae_3}, R2 3: {r2_3}')

RMSE 3: 72.80911781581105, MAE 3: 38.34705467042321, R2 3: -0.07774374967213671


# 8. Random Forest Regressor

In [164]:
columnas_eliminar = []
for columna in X_train.columns:
    if columna.startswith('sexo'):
        columnas_eliminar.append(columna)
    elif columna.startswith('dia'):
        columnas_eliminar.append(columna)
print(columnas_eliminar)

['sexo_Femenino', 'sexo_Masculino', 'sexo_No especificado', 'dia_solicitud_Friday', 'dia_solicitud_Monday', 'dia_solicitud_Saturday', 'dia_solicitud_Sunday', 'dia_solicitud_Thursday', 'dia_solicitud_Tuesday', 'dia_solicitud_Wednesday']


In [165]:
X_train_final = X_train.drop(columns=columnas_eliminar)
X_test_final = X_test.drop(columns=columnas_eliminar)

In [170]:
param_dict = {
    'n_estimators' : [100, 200, 300, 500],
    'max_depth' : [10, 15, 20, 25],
    'min_samples_split' : [2, 5, 10, 20],
    'min_samples_leaf' : [1, 2, 5, 10]
}

In [171]:
rfr = RandomForestRegressor(max_features='sqrt', n_jobs=-1, random_state=0)
random_search = RandomizedSearchCV(
    estimator=rfr,
    param_distributions=param_dict,
    n_iter=20,
    cv=3,
    scoring='r2',
    n_jobs=-1,
    random_state=0,
    verbose=2
)

random_search.fit(X=X_train_final, y=y_train_log)

print(random_search.best_params_)
print(random_search.best_score_)

Fitting 3 folds for each of 20 candidates, totalling 60 fits
[CV] END max_depth=20, min_samples_leaf=5, min_samples_split=10, n_estimators=300; total time= 6.9min
[CV] END max_depth=20, min_samples_leaf=5, min_samples_split=10, n_estimators=300; total time= 6.9min
[CV] END max_depth=20, min_samples_leaf=2, min_samples_split=20, n_estimators=300; total time= 6.9min
[CV] END max_depth=20, min_samples_leaf=2, min_samples_split=20, n_estimators=300; total time= 6.9min
[CV] END max_depth=20, min_samples_leaf=2, min_samples_split=20, n_estimators=300; total time= 6.9min
[CV] END max_depth=15, min_samples_leaf=2, min_samples_split=2, n_estimators=500; total time= 8.7min
[CV] END max_depth=15, min_samples_leaf=2, min_samples_split=2, n_estimators=500; total time= 8.7min
[CV] END max_depth=15, min_samples_leaf=2, min_samples_split=2, n_estimators=500; total time= 8.7min
[CV] END max_depth=15, min_samples_leaf=5, min_samples_split=5, n_estimators=200; total time= 3.3min
[CV] END max_depth=15, mi

/Users/sergiobenhumea/miniforge3/envs/ds_full/lib/python3.11/site-packages/joblib/externals/loky/process_executor.py:782: UserWarning: A worker stopped while some jobs were given to the executor. This can be caused by a too short worker timeout or by a memory leak.
  warnings.warn(


[CV] END max_depth=20, min_samples_leaf=2, min_samples_split=10, n_estimators=200; total time= 3.9min
[CV] END max_depth=15, min_samples_leaf=1, min_samples_split=10, n_estimators=200; total time= 3.1min
[CV] END max_depth=20, min_samples_leaf=2, min_samples_split=20, n_estimators=200; total time= 4.3min
[CV] END max_depth=20, min_samples_leaf=2, min_samples_split=20, n_estimators=200; total time= 4.4min
[CV] END max_depth=20, min_samples_leaf=5, min_samples_split=10, n_estimators=500; total time=10.2min
[CV] END max_depth=20, min_samples_leaf=2, min_samples_split=20, n_estimators=200; total time= 4.4min
[CV] END max_depth=15, min_samples_leaf=5, min_samples_split=2, n_estimators=200; total time= 3.4min
[CV] END max_depth=20, min_samples_leaf=5, min_samples_split=10, n_estimators=500; total time=10.1min
[CV] END max_depth=20, min_samples_leaf=5, min_samples_split=10, n_estimators=500; total time=10.0min
[CV] END max_depth=25, min_samples_leaf=1, min_samples_split=20, n_estimators=500; 

In [174]:
y_pred_rf = np.expm1(random_search.best_estimator_.predict(X_test_final))
rmse_rfr = root_mean_squared_error(y_true=y_test, y_pred=y_pred_rf)
mae_rfr = mean_absolute_error(y_true=y_test, y_pred=y_pred_rf)
r2_rfr = r2_score(y_true=y_test, y_pred=y_pred_rf)
print(f'RMSE RFR: {rmse_rfr}, MAE RFR: {mae_rfr}, R2 RFR: {r2_rfr}')

RMSE RFR: 69.8672304639431, MAE RFR: 35.58944971042609, R2 RFR: 0.007590229841611507


In [175]:
import plotly.express as px

residuales_rf = y_test - y_pred_rf

fig = px.histogram(residuales_rf, nbins=100, title='Distribución de residuales — Random Forest')
fig.update_layout(xaxis_title='Residual (y_test - y_pred)', yaxis_title='Frecuencia')
fig.show()

# 9. XGBoost